In [5]:
import pandas as pd
import re

In [8]:
# 1. VALOR APROVADO POR ANO/MÊS DE ATENDIMENTO
# ─────────────────────────────────────────────────────────────────────────────
df_valor_mes = pd.read_csv(
    "sia_cnv_qace094212177_19_126_92.csv",
    encoding="latin1",
    sep=";",
    skiprows=3,
)
 
# Renomear colunas para facilitar manipulação
df_valor_mes.columns = ["ano_mes", "valor_aprovado"]
 
# Separar linhas de totais anuais (sem ".." no início) das mensais
df_valor_mes["nivel"] = df_valor_mes["ano_mes"].apply(
    lambda x: "anual" if not str(x).startswith("..") else "mensal"
)
 
# Limpar valor: trocar vírgula por ponto e converter para float
df_valor_mes["valor_aprovado"] = (
    df_valor_mes["valor_aprovado"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)
 
# Extrair o ano das linhas mensais para facilitar filtros futuros
df_valor_mes["ano"] = df_valor_mes["ano_mes"].str.extract(r"(\d{4})").astype("Int64")
 
print("Arquivo 1 carregado:", df_valor_mes.shape)
print(df_valor_mes.head(6))
print()


Arquivo 1 carregado: (130, 4)
           ano_mes  valor_aprovado   nivel   ano
0             2014    2.351270e+06   anual  2014
1   ..Outubro/2014    1.218440e+05  mensal  2014
2  ..Novembro/2014    1.428088e+05  mensal  2014
3  ..Dezembro/2014    2.086617e+06  mensal  2014
4             2015    5.724245e+08   anual  2015
5   ..Janeiro/2015    4.616483e+07  mensal  2015



In [9]:
# 2. QUANTIDADE APROVADA POR MUNICÍPIO E ANO DE PROCESSAMENTO
# ─────────────────────────────────────────────────────────────────────────────
df_municipio = pd.read_csv(
    "sia_cnv_qace094359177_19_126_92.csv",
    encoding="latin1",
    sep=";",
    skiprows=3,
)
 
# Identificar colunas de ano e total
anos_cols = [c for c in df_municipio.columns if c.strip().isdigit()]
df_municipio.columns = ["municipio"] + anos_cols + ["total"]
 
# Extrair código IBGE e nome do município
df_municipio["cod_ibge"] = df_municipio["municipio"].str.extract(r"^(\d+)")
df_municipio["nome_municipio"] = df_municipio["municipio"].str.replace(
    r"^\d+\s*", "", regex=True
).str.strip()
df_municipio = df_municipio.drop(columns=["municipio"])
 
# Remover linha de total geral (última linha geralmente)
df_municipio = df_municipio[df_municipio["cod_ibge"].notna()].copy()
 
print(" Arquivo 2 carregado:", df_municipio.shape)
print(df_municipio.head(4))
print()
 


 Arquivo 2 carregado: (184, 12)
       2015       2016       2017      2018      2019      2020      2021  \
0  149266.0   152921.0   106266.0   14659.0   37631.0   37383.0   43891.0   
1  242728.0   249150.0   464276.0  155623.0   84783.0   53131.0   49620.0   
2  780288.0  1399036.0  1304505.0  843912.0  242712.0  188585.0  286459.0   
3  287419.0   261456.0   253037.0  130776.0  160517.0   96029.0   95958.0   

       2022      2023      total cod_ibge nome_municipio  
0   49861.0   53277.0   645155.0   230010        ABAIARA  
1   69204.0   76079.0  1444594.0   230015        ACARAPE  
2  443886.0  526968.0  6016351.0   230020         ACARAU  
3  146301.0  183443.0  1614936.0   230030       ACOPIARA  



In [10]:
# 3. VALOR APROVADO POR PROCEDIMENTO E ANO DE PROCESSAMENTO
# ─────────────────────────────────────────────────────────────────────────────
df_proc = pd.read_csv(
    "sia_cnv_qace094828177_19_126_92.csv",
    encoding="latin1",
    sep=";",
    skiprows=3,
)
 
anos_cols_p = [c for c in df_proc.columns if c.strip().isdigit()]
df_proc.columns = ["procedimento"] + anos_cols_p + ["total"]
 
# Extrair código e nome do procedimento
df_proc["cod_procedimento"] = df_proc["procedimento"].str.extract(r"^(\d+)")
df_proc["nome_procedimento"] = df_proc["procedimento"].str.replace(
    r"^\d+\s*", "", regex=True
).str.strip()
df_proc = df_proc.drop(columns=["procedimento"])
 
# Converter valores (formato BR: "1.234,56" → float)
def br_to_float(s):
    return (
        str(s)
        .replace(".", "")
        .replace(",", ".")
        .replace("-", "0")
        .strip()
    )
 
for col in anos_cols_p + ["total"]:
    df_proc[col] = df_proc[col].apply(br_to_float).astype(float)
 
# Remover linha de total geral
df_proc = df_proc[df_proc["cod_procedimento"].notna()].copy()
 
print("Arquivo 3 carregado:", df_proc.shape)
print(df_proc.head(4))
print()


Arquivo 3 carregado: (1766, 12)
        2015       2016       2017       2018       2019      2020       2021  \
0  174555.00  121707.90  107044.20  113880.60  128250.00  87318.00  159030.00   
1   15729.00   21141.00   30075.00   28980.00   29421.00  26091.00   31383.00   
2    8283.94   19343.94   31366.16   29176.28   29342.18  31388.28   37526.58   
3    1398.00     698.00       0.00       0.00       0.00      0.00       0.00   

        2022       2023       total cod_procedimento  \
0  243315.90  368296.20  1503397.80       0101010028   
1   31011.00   37170.00   251001.00       0101040032   
2   35469.42   42614.18   264510.96       0101040040   
3       0.00       0.00     2096.00       0102010056   

                                   nome_procedimento  
0  ATIVIDADE EDUCATIVA / ORIENTACAO EM GRUPO NA A...  
1      COLETA EXTERNA DE LEITE MATERNO (POR DOADORA)  
2      PASTEURIZACAO DO LEITE HUMANO (CADA 5 LITROS)  
3        ATIVIDADES EDUCATIVAS PARA O SETOR REGULADO  



In [11]:
# 4. CRIAR VISÃO UNIFICADA POR ANO (chave comum entre os três arquivos)
# ─────────────────────────────────────────────────────────────────────────────
 
# 4a. Totais anuais do arquivo 1 (apenas linhas de nível anual)
df_anual_valor = df_valor_mes[df_valor_mes["nivel"] == "anual"][
    ["ano_mes", "valor_aprovado"]
].copy()
df_anual_valor.columns = ["ano", "valor_aprovado_total"]
df_anual_valor["ano"] = df_anual_valor["ano"].str.extract(r"(\d{4})").astype(int)
 
# 4b. Totais anuais do arquivo 2 (soma de municípios por ano)
df_qtd_por_ano = (
    df_municipio[anos_cols]
    .sum()
    .reset_index()
)
df_qtd_por_ano.columns = ["ano", "qtd_aprovada_total"]
df_qtd_por_ano["ano"] = df_qtd_por_ano["ano"].astype(int)
 
# 4c. Totais anuais do arquivo 3 (soma de procedimentos por ano)
df_valor_proc_ano = (
    df_proc[anos_cols_p]
    .sum()
    .reset_index()
)
df_valor_proc_ano.columns = ["ano", "valor_proc_total"]
df_valor_proc_ano["ano"] = df_valor_proc_ano["ano"].astype(int)
 
# Junção dos três resumos anuais
df_unificado = (
    df_anual_valor
    .merge(df_qtd_por_ano, on="ano", how="outer")
    .merge(df_valor_proc_ano, on="ano", how="outer")
    .sort_values("ano")
    .reset_index(drop=True)
)
 
print("=" * 60)
print(" DATAFRAME UNIFICADO POR ANO")
print("=" * 60)
print(df_unificado.to_string(index=False))
print()


ValueError: cannot convert float NaN to integer